In [1]:
from google.colab import files
up = files.upload()

Saving 100_Unique_QA_Dataset.csv to 100_Unique_QA_Dataset.csv


In [2]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("/content/100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


* S1) Tokenize

In [101]:
import re
def tokenizeText(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return text.split()

In [89]:
tokenizeText("What is AI?")

['what', 'is', 'ai']

* S2) Vocab

In [102]:
def buildVocab(row):
  tokenizedQues = tokenizeText(row['question'])
  tokenizedAns = tokenizeText(row['answer'])
  mergedTokens = tokenizedQues + tokenizedAns
  for token in mergedTokens:
    if token not in vocab:
      vocab[token] = len(vocab)


In [103]:
vocab = {'<UNK>': 0}
df.apply(buildVocab, axis=1)


,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [93]:
len(vocab)

324

In [104]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper': 16,
 'lee': 17,
 'largest': 18,
 'planet': 19,
 'in': 20,
 'our': 21,
 'solar': 22,
 'system': 23,
 'jupiter': 24,
 'boiling': 25,
 'point': 26,
 'water': 27,
 'celsius': 28,
 '100': 29,
 'painted': 30,
 'mona': 31,
 'lisa': 32,
 'leonardo': 33,
 'da': 34,
 'vinci': 35,
 'square': 36,
 'root': 37,
 '64': 38,
 '8': 39,
 'chemical': 40,
 'symbol': 41,
 'for': 42,
 'gold': 43,
 'au': 44,
 'which': 45,
 'year': 46,
 'did': 47,
 'world': 48,
 'war': 49,
 'ii': 50,
 'end': 51,
 '1945': 52,
 'longest': 53,
 'river': 54,
 'nile': 55,
 'japan': 56,
 'tokyo': 57,
 'developed': 58,
 'theory': 59,
 'relativity': 60,
 'albert': 61,
 'einstein': 62,
 'freezing': 63,
 'fahrenheit': 64,
 '32': 65,
 'known': 66,
 'as': 67,
 'red': 68,
 'mars': 69,
 'author': 70,
 '1984': 71,
 'george': 72

In [105]:
# Converting Words Into Numerical Tokens
def textToIndices(text, vocab):
  indexedText = []
  for token in tokenizeText(text):
    if token in vocab:
      indexedText.append(vocab[token])
    else:
      indexedText.append(vocab['<UNK>'])
  return indexedText

In [106]:
textToIndices("What is AI", vocab)

[1, 2, 0]

In [107]:
import torch

In [108]:
from torch.utils.data import Dataset , DataLoader

In [109]:
class CustomDataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, idx):
    row = self.df.iloc[idx]
    question = textToIndices(row['question'], self.vocab)
    answer = textToIndices(row['answer'], self.vocab)

    return torch.tensor(question), torch.tensor(answer)

In [110]:
dataset = CustomDataset(df, vocab)

In [111]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [112]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [113]:
for ques, ans in dataloader:
  print(ques ,ans[0])

tensor([[ 10, 101,   3, 102]]) tensor([103, 104, 105])
tensor([[ 10, 150,   3, 151, 182,   5,   3,  75, 183]]) tensor([ 72, 184])
tensor([[  1,   2,   3,   4,   5, 123]]) tensor([124])
tensor([[ 1,  2,  3, 36, 37,  5, 38]]) tensor([39])
tensor([[ 10,  30, 140, 141]]) tensor([142])
tensor([[ 45, 329,   2,  66,  67,   3, 330,   5, 331]]) tensor([332])
tensor([[ 1,  2,  3,  4,  5, 78]]) tensor([79])
tensor([[ 1,  2,  3, 74,  5, 56]]) tensor([271])
tensor([[ 1,  2,  3, 97, 98, 99]]) tensor([100])
tensor([[ 45, 261, 262, 128, 263, 264]]) tensor([265])
tensor([[ 1,  2,  3, 53, 54, 20,  3, 48]]) tensor([55])
tensor([[ 45,  91,  92, 252, 253,  20,  42, 254]]) tensor([255])
tensor([[10,  2,  3, 70,  5, 71]]) tensor([72, 73])
tensor([[ 10,  11, 167, 168, 169]]) tensor([170, 171])
tensor([[  1,   2,   3,   4,   5, 145]]) tensor([146])
tensor([[83, 84, 85, 86, 87, 88, 89]]) tensor([90])
tensor([[  1,   2,   3, 156, 157,  20, 158]]) tensor([159])
tensor([[  1,   2,   3, 245,   5, 246]]) tensor([141

In [114]:
import torch.nn as nn

In [115]:
class Model(nn.Module):
    def __init__(self, vocabSize):
        super().__init__()
        self.embed = nn.Embedding(vocabSize, 50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, vocabSize)

    def forward(self, question):
      embeddedQues = self.embed(question)
      hidden, final = self.rnn(embeddedQues)
      output = self.fc(final.squeeze(0))
      return output


In [122]:
learningRate = 0.001
epochs = 50

In [117]:
RNNModel = Model(len(vocab))

In [118]:
lossFunc = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(RNNModel.parameters(), lr=learningRate)

In [123]:
for epoch in range(epochs):
  totalLoss = 0
  for question, answer in dataloader:
    optimizer.zero_grad()
    output = RNNModel(question)

    target = answer[0][0]
    loss = lossFunc(output, target.unsqueeze(0))

    loss.backward()

    optimizer.step()

    totalLoss = totalLoss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {totalLoss/len(dataloader):4f}")

Epoch: 1, Loss: 0.000009
Epoch: 2, Loss: 0.000009
Epoch: 3, Loss: 0.000008
Epoch: 4, Loss: 0.000008
Epoch: 5, Loss: 0.000008
Epoch: 6, Loss: 0.000007
Epoch: 7, Loss: 0.000007
Epoch: 8, Loss: 0.000007
Epoch: 9, Loss: 0.000006
Epoch: 10, Loss: 0.000006
Epoch: 11, Loss: 0.000006
Epoch: 12, Loss: 0.000006
Epoch: 13, Loss: 0.000005
Epoch: 14, Loss: 0.000005
Epoch: 15, Loss: 0.000005
Epoch: 16, Loss: 0.000005
Epoch: 17, Loss: 0.000004
Epoch: 18, Loss: 0.000004
Epoch: 19, Loss: 0.000004
Epoch: 20, Loss: 0.000004
Epoch: 21, Loss: 0.000004
Epoch: 22, Loss: 0.000004
Epoch: 23, Loss: 0.000003
Epoch: 24, Loss: 0.000003
Epoch: 25, Loss: 0.000003
Epoch: 26, Loss: 0.000003
Epoch: 27, Loss: 0.000003
Epoch: 28, Loss: 0.000003
Epoch: 29, Loss: 0.000003
Epoch: 30, Loss: 0.000002
Epoch: 31, Loss: 0.000002
Epoch: 32, Loss: 0.000002
Epoch: 33, Loss: 0.000002
Epoch: 34, Loss: 0.000002
Epoch: 35, Loss: 0.000002
Epoch: 36, Loss: 0.000002
Epoch: 37, Loss: 0.000002
Epoch: 38, Loss: 0.000002
Epoch: 39, Loss: 0.00

In [124]:
def predict(model, question, threshold=0.5):

  numericalQuestion = textToIndices(question, vocab)
  question_tensor = torch.tensor(numericalQuestion).unsqueeze(0)
  output = RNNModel(question_tensor)

  probs = torch.nn.functional.softmax(output, dim=1)
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [126]:
predict(RNNModel, "What is the largest planet in our solar system?")

jupiter


In [127]:
predict(RNNModel, "Who wrote 'To Kill a Mockingbird'?")

harper
